# Глубинное обучение 1 / Введение в глубинное обучение, ФКН ВШЭ

## Домашнее задание 3: LoRA-адаптеры

### Общая информация

Оценка после штрафа после мягкого дедлайна вычисляется по формуле $M_{\text{penalty}} = M_{\text{full}} \cdot 0.85^{t/1440}$, где $M_{\text{full}}$ — полная оценка за работу без учета штрафа, а $t$ — время в минутах, прошедшее после мягкого дедлайна (округление до двух цифр после запятой). Таким образом, спустя первые сутки после мягкого дедлайна вы не можете получить оценку выше 12.75, а если сдать через четыре дня после мягкого дедлайна, то ваш максимум — 7.83 балла.

### Оценивание и штрафы

Максимально допустимая оценка за работу — 15 баллов. Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

### О задании

В этом задании вам предстоит познакомиться с [`LoRA`](https://arxiv.org/pdf/2106.09685) адаптерами для эффективного дообучения нейронных сетей в ситуации ограниченных вычислительных ресурсов и больших моделей, которые могут напрямую не влезать в оперативную память GPU, как с самым распространённым видом [PEFT](https://huggingface.co/docs/peft/en/index). В качестве данных будем использовать набор из 120 тысяч анекдотов из первой части задания (архив `jokes.txt.zip`, который доступен по [ссылке](https://disk.yandex.com/d/fjt5xICH-ukEEA))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## Задание 1: Предварительные вопросы (1 балла)

Во всём этом задании мы будем следить за размером потребляемой видеопамяти. Если вы делаете работу на T4 в Google Colab, то её у вас должно быть около 15ГБ.

Письменно ответьте на следующие вопросы:

- в каком базовом формате хранятся веса в pytorch? сколько памяти на видеокарте они занимают?
- сколько занимает памяти модель размера `N` миллионов весов без кастомной квантизации? `N` миллиардов весов?
- сколько занимают видеопамяти те же модели в `int8` квантизации? а в `int4` квантизации?
- от чего, помимо числа параметров, зависит объём видеопамяти при обучении? как он зависит от `batch_size` и длины последовательности? что именно в памяти растёт при переходе от `batch_size = 1` к `batch_size = 128`, а что остаётся неизменным?
- почему при инференсе модели (под `torch.no_grad()`) зависимость от `batch_size` гораздо слабее, чем при обучении?
- пусть у нашей модели `N` миллиардов весов, зададим `batch_size = 1`. если мы не используем `gradient checkpointing`, сколько нужно видеопамяти для оптимизации этой модели на одной видеокарте с помощью `SGD` (с `momentum` и без), `Adam`? почему?
- опишите своими словами, как работает `gradient checkpointing` и чем он помогает в обучении?

Дальше в ходе домашки мы проверим часть из ваших ответов на практике.

Ваши ответы здесь:



## Задание 2: Загрузка моделей (1 балл)

Ваша - задача - скачать с разными конфигами квантизации модель `ai-forever/rugpt3large_based_on_gpt2` из библиотеки `transformers`. Современный способ загрузки квантизованных моделей - указание конфига `BitsAndBytesConfig`. Покажите разницу между стандартной моделью, а также квантизованными в `int8` и `int4`.

Загрузите модель с разными конфигами и выведите занимаемую память при помощи функции ниже. Отличаются ли выводы от того, что вы предполагали в пункте 1? Ответ прокомментируйте.

В конце сделайте таблицу в формате:

dtype | теоретический размер | `get_memory_footprint` | `memory_allocated` | отношение к fp32
--|--|--|--|--
base (`float32`) | | | |
`int8` | | | |
`int4` | | | |

In [ ]:
def print_memory_usage(model):
    print(f"model:     {model.get_memory_footprint() / 2**30:.2f} GiB")
    print(f"allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
    print(f"peak:      {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# usage:
# model = AutoModelForCausalLM.from_pretrained(...)
# print_memory_usage(model)
# del model
# free_memory()

## Задание 3: Дообучение модели целиком с помощью `SGD`, `Adam` (1 балл)

Теперь, когда мы познакомились с квантизованными версиями моделей, загрузите модель в `fp32`. В этом пункте вам нужно экспериментально найти такую границу по `batch_size`, что при `batch_size = B` модель обучается, а при `batch_size = 2 * B` обучение падает. В данном пункте использовать `gradient checkpointing` запрещено.

Для этого загрузите датасет, оберните его в `DataLoader` и попробуйте прогнать 100 шагов одной эпохи с выбранным размером батча. Сделайте это для `Adam`, `SGD` без `momentum` и с ним. Итого будет 3 эксперимента.

Если вы работаете в колабе на `T4` и всё сделали правильно, то в одном из трёх случаев вы скорее всего даже не сможете прогнать модель с `batch_size = 1`, а в других случаях при увеличении размера вы довольно скоро получите `OOM` ошибку и всё упадёт. Для каждого из трёх упомянутых случаев опишите увиденные результаты. Почему так получилось?

В этом задании необязательно прогонять всю эпоху целиком. К тому же скорее всего даже одна эпоха у вас займёт больше часа. Если так, возьмите вместо всей обучающей выборки только её 10-20% - вам достаточно будет понять время, которое бы у вас занял прогон всего датасета для оценки скорости.

Итого:
- загрузите модель в `fp32`
- напишите функцию обучения
- покажите, что обучение действительно падает по `OOM` и зафиксируйте параметры, при которых это наблюдается
- для максимальных значений `batch_size` попробуйте прогнать эпоху (может быть на 10-20% выборки)
- оцените скорость прогона 1 эпохи целиком на всей обучающей выборке для каждого из оптимизаторов, прогон которых влез в память
- в конце пункта опишите результаты в табличке:

`4N` - столько байт весит модель с `N` обучаемых параметров в модели. В колонке `теория` вместо `?` поставьте натуральное число
оптимизатор| макс. batch_size| теория (веса + градиенты + состояния)| пик (замер)| разница
--|--|--|--| --
SGD| | ?N| |
SGD + momentum| | ?N| |
Adam| | ?N| |

In [ ]:
from torch.utils.data import DataLoader

MAX_LEN = 128 # set to 95% of dataset sapmles len
BATCH_SIZE = 1

def encode_jokes(texts, tokenizer, max_len=MAX_LEN):
    # <s> joke </s>
    ids = tokenizer([t.strip() for t in texts], add_special_tokens=False,
                    truncation=True, max_length=max_len - 2)['input_ids']
    return [[tokenizer.bos_token_id] + x + [tokenizer.eos_token_id] for x in ids if x]

def make_collate(pad_id):
    # pads a batch to MAX_LEN
    def collate(batch):
        input_ids = torch.full((len(batch), MAX_LEN), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch), MAX_LEN), dtype=torch.long)
        for i, x in enumerate(batch):
            input_ids[i, :len(x)] = torch.tensor(x)
            attention_mask[i, :len(x)] = 1
        labels = input_ids.masked_fill(attention_mask == 0, -100)
        return input_ids, attention_mask, labels
    return collate


collate = make_collate(tokenizer.pad_token_id)
train_loader = DataLoader(encode_jokes(train_texts, tokenizer), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)

## Задание 4: `LoRA`-адаптер своими руками (3 балла)

Обязательно перезагрузите сессию и сделайте нужные импорты снова, начиная с этого места, чтобы остались выходы ячеек с предыдущих шагов (перезагрузка сессии понадобится для полной очистки видеопамяти).

В прошлом задании вы убедились, что при дообучении модели по мере её роста узким местом может стать видеопамять, поскольку нужно одновременно хранить:

- веса модели
- градиенты
- накладные расходы оптимизатора

То есть только для того, чтобы сделать шаг оптимизатора возможным, приходится хранить порой до 4-кратного объёма памяти от размера самой модели. Всё это в случае обучения модели целиком. Но если при этом обучать только часть весов модели, а остальные не трогать, объём требуемой памяти для градиентов и накладных расходов оптимизатора резко снижается, и будет линейно зависеть от числа обучаемых параметров.

Так, написанный подавляющим большинством из вас слой `LowRankLinear` из первой домашки чем-то очень напоминает реализацию `LoRA`-адаптера.

Описание задачи:

- перезагрузить сессию, загрузить модель в `fp32`
- реализовать класс `LoRALinear` с поддержкой инициализации от `nn.Linear` и `Conv1D` из `transformers.pytorch_utils`
    - класс наследуется от `nn.Module`
    - принимает на вход слой линейного преобразования `x -> xW + b` и поддерживает два типа таких слоёв (см. выше, несмотря на название, `Conv1D` - это обычный линейный слой, который используется в архитектуре `GPT-2`, обратите внимание, как в нём реализовано хранение матрицы весов)
    - отдельно задаётся параметр `r` ранга адаптера
    - сделайте инициализацию матриц, как в [статье](https://arxiv.org/pdf/2106.09685)
    - не используется масштабирование на $\frac{\alpha}{r}$ из оригинальной статьи
    - делайте добавку только для матрицы, на которую умножаются веса, игнорируя (оставляя, если он есть, вектор `bias`)
    - реализация должна учитывать отсутствие оверхеда на инференсе, а также не должна удваивать объём требуемой памяти - будьте аккуратны с реализацией
- заморозить все веса модели
- замените на `LoRALinear` все слои линейного преобразования внутри трансформерных блоков модели: в attention и MLP
- выходной слой `lm_head` не трогайте
- зафиксируйте оптимизатор, который вам больше нравится (обоснуйте свой выбор)
- зафиксируйте соответствующие ему `BATCH_SIZE` и `MAX_LEN`
- самостоятельно выберите `r` и обоснуйте свой выбор
- прогнать одну эпоху целиком (если позволяет видеокарта и время - можно 3 эпохи. если время и видеокарта не позволяют - возьмите 10-20% выборки и 1 эпоху) и вывести:
    - число обучаемых параметров
    - пиковое потребление видеопамяти
    - время прогона целой эпохи
    - train loss, val loss
    - perplexity
- использовать `gradient checkpointing` запрещено

## Задание 5: `QLoRA` (1 балл)

В задании 4 мы сделали дообучение на 1 эпохе большой модели с собственным адаптером. Теперь сделайте полноценное дообучение квантизованной модели с `QLoRA` из библиотеки `peft`.

- перед заданием настоятельно рекомендуется снова перезагрузить сессию для очистки памяти на gpu
- зафиксируйте число эпох обучения, `lr`, `batch_size` на свой вкус
- замерьте скорость одной эпохи, пиковое потребление видеопамяти и сравните с обычной `LoRA` из предыдущего пункта. есть ли ускорение по времени? почему его может не быть?
- на каждой эпохе обучения отслеживайте loss, а также семплируйте по 3 анекдота
- используйте те же `r`, `target modules` и `batch_size`, `max_len` что в задании 4
- используйте `lora_alpha = r`
- как и в прошлом пункте, прогоните одну эпоху целиком (если позволяет видеокарта и время - можно 3 эпохи. если время и видеокарта не позволяют - возьмите 10-20% выборки и 1 эпоху) и выведите:
    - число обучаемых параметров
    - пиковое потребление видеопамяти
    - время прогона целой эпохи
    - train loss, val loss
    - perplexity
- в конце оставьте самый смешной анекдот на ваш вкус, который получилось сгенерировать обученной моделью

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

## Задание 6 (Бонус): Gradient Checkpointing (1 балл)

Возьмите свой же сетап из прошлого задания и поэкспериментируйте с `gradient checkpointing`. Делается это с помощью функции [prepare_model_for_kbit_training](https://huggingface.co/docs/peft/v0.21.0/en/package_reference/peft_model#peft.prepare_model_for_kbit_training).

Ваша задача - дообучить квантизованную модель с `QLoRA` с включённым `gradient checkpointing` и показать различие между включенным и выключенным значением этого параметра.